# Tutorial 6: Mixed Precision Quantization Search with Mase and Optuna

In this tutorial, we'll see how Mase can be integrated with Optuna, the popular hyperparameter optimization framework, to search for a Bert model optimized for sequence classification on the IMDb dataset. We'll take the Optuna-generated model and import it into Mase, then run the CompressionPipeline to prepare the model for edge deployment by quantizing and pruning its weights.

As we'll see, running Architecture Search with Mase/Optuna involves the following steps.

1. **Define the search space**: this is a dictionary containing the range of values for each parameter at each layer in the model.

2. **Write the model constructor**: this is a function which uses Optuna utilities to sample a model from the search space, and constructs the model using transformers from_config class method.

3. **Write the objective function**: this function calls on the model constructor defined in Step 2 and defines the training/evaluation setup for each search iteration.

4. **Go!** Choose an Optuna sampler, create a study and launch the search.

In [1]:
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

## Importing the model

If you are starting from scratch, you can load the Bert checkpoint directly from HuggingFace.

In [5]:
from transformers import AutoModel

model = AutoModel.from_pretrained(checkpoint)

If you have previously ran the tutorial on Neural Architecture Search (NAS), run the following cell to import the best model obtained from the search process.

In [2]:
from pathlib import Path
import dill

with open(f"best_tpe_model.pkl", "rb") as f:
    base_model = dill.load(f)

/Users/nyalpatel/anaconda3/envs/mase/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


First, fetch the dataset using the `get_tokenized_dataset` utility.

In [3]:
from chop.tools import get_tokenized_dataset

dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.


## 1. Defining the Search Space

We'll start by defining a search space, i.e. enumerating the possible combinations of hyperparameters that Optuna can choose during search. We'll explore the following range of values for the model's hidden size, intermediate size, number of layers and number of heads.

In [5]:
import torch
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockMinifloat,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
    LinearBinaryResidualSign,
)

search_space = {
    "linear_layer_choices": [
        torch.nn.Linear,
        LinearInteger,
    ],
}

## 2. Writing a Model Constructor

We define the following function, which will get called in each iteration of the search process. The function is passed the `trial` argument, which is an Optuna object that comes with many functionalities - see the [Trial documentation](https://optuna.readthedocs.io/en/stable/reference/trial.html) for more details. Here, we use the `trial.suggest_categorical` function, which triggers the chosen sampler to choose a layer type. The suggested integer is the index into the search space for each parameter, which we defined in the previous cell.

In [6]:
from chop.tools.utils import deepsetattr
from copy import deepcopy


def construct_model(trial):

    # Fetch the model
    trial_model = deepcopy(base_model)

    # Quantize layers according to optuna suggestions
    for name, layer in trial_model.named_modules():
        if isinstance(layer, torch.nn.Linear):
            new_layer_cls = trial.suggest_categorical(
                f"{name}_type",
                search_space["linear_layer_choices"],
            )

            if new_layer_cls == torch.nn.Linear:
                continue

            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }

            # If the chosen layer is integer, define the low precision config
            if new_layer_cls == LinearInteger:
                kwargs["config"] = {
                    "data_in_width": 8,
                    "data_in_frac_width": 4,
                    "weight_width": 8,
                    "weight_frac_width": 4,
                    "bias_width": 8,
                    "bias_frac_width": 4,
                }
            # elif... (other precisions)

            # Create the new layer (copy the weights)
            new_layer = new_layer_cls(**kwargs)
            new_layer.weight.data = layer.weight.data

            # Replace the layer in the model
            deepsetattr(trial_model, name, new_layer)

    return trial_model

## 3. Defining the Objective Function

Next, we define the objective function for the search, which gets called on each trial. In each trial, we create a new model instace with chosen hyperparameters according to the defined sampler. We then use the `get_trainer` utility in Mase to run a training loop on the IMDb dataset for a number of epochs. Finally, we use `evaluate` to report back the classification accuracy on the test split.

In [7]:
from chop.tools import get_trainer
import random


def objective(trial):

    # Define the model
    model = construct_model(trial)

    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    trainer.train()
    eval_results = trainer.evaluate()

    trial.set_user_attr("model", model)

    return eval_results["eval_accuracy"]

## 4. Launching the Search

Optuna provides a number of samplers, for example:

* **GridSampler**: iterates through every possible combination of hyperparameters in the search space
* **RandomSampler**: chooses a random combination of hyperparameters in each iteration
* **TPESampler**: uses Tree-structured Parzen Estimator algorithm to choose hyperparameter values.

You can define the chosen sampler by simply importing from `optuna.samplers` as below.

In [8]:
from optuna.samplers import GridSampler, RandomSampler, TPESampler

sampler = RandomSampler()

With all the pieces in place, we can launch the search as follows. The number of trials is set to 1 so you can go get a coffee for 10 minutes, then proceed with the tutorial. However, this will essentially be a random model - for better results, set this to 100 and leave it running overnight!

In [9]:
import optuna

study = optuna.create_study(
    direction="maximize",
    study_name="bert-tiny-nas-study",
    sampler=sampler,
)

study.optimize(
    objective,
    n_trials=1,
    timeout=60 * 60 * 24,
)

[I 2025-01-31 16:27:59,251] A new study created in memory with name: bert-tiny-nas-study
/Users/nyalpatel/anaconda3/envs/mase/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'torch.nn.modules.linear.Linear'> which is of type type.
  warnings.warn(message)
/Users/nyalpatel/anaconda3/envs/mase/lib/python3.11/site-packages/optuna/distributions.py:515: UserWarning: Choices for a categorical distribution should be a tuple of None, bool, int, float and str for persistent storage but contains <class 'chop.nn.quantized.modules.linear.LinearInteger'> which is of type type.
  warnings.warn(message)
/Users/nyalpatel/Library/CloudStorage/OneDrive-Personal/Uni/Imperial MSC/Modules/Advanced Deep Learning/Labs/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init_

Step,Training Loss
500,0.341300
1000,0.307700
1500,0.330400
2000,0.329300
2500,0.301800
3000,0.334900


[W 2025-01-31 16:38:02,439] Trial 0 failed with parameters: {'bert.encoder.layer.0.attention.self.query_type': <class 'chop.nn.quantized.modules.linear.LinearInteger'>, 'bert.encoder.layer.0.attention.self.key_type': <class 'chop.nn.quantized.modules.linear.LinearInteger'>, 'bert.encoder.layer.0.attention.self.value_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.attention.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.intermediate.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.0.output.dense_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.query_type': <class 'chop.nn.quantized.modules.linear.LinearInteger'>, 'bert.encoder.layer.1.attention.self.key_type': <class 'torch.nn.modules.linear.Linear'>, 'bert.encoder.layer.1.attention.self.value_type': <class 'chop.nn.quantized.modules.linear.LinearInteger'>, 'bert.encoder.layer.1.attention.output.dense_type': <class 

KeyboardInterrupt: 

Attempy 3 - Adding acerage number of bits as a cost to the loos function

In [6]:
import random
import dill
from copy import deepcopy
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler

from transformers import AutoModel
from chop.tools import get_tokenized_dataset, get_trainer
from chop.tools.utils import deepsetattr

# Import supported linear quantizers.
from torch import nn
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    # LinearBlockMinifloat,  # Uncomment if desired.
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
    # Exclude LinearBinaryResidualSign.
)
from chop.passes.graph.analysis.quantization.calculate_avg_bits import calculate_avg_bits_mg_analysis_pass

from chop.ir.graph import MaseGraph
import chop.passes as passes

import os
os.makedirs("./logs", exist_ok=True)

# ------------------------------------------------------------------------------
# Configuration: Checkpoints, dataset, etc.
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

# Load the floating-point base model.
model = AutoModel.from_pretrained(checkpoint)
with open("best_tpe_model.pkl", "rb") as f:
    base_model = dill.load(f)

# Get tokenized dataset and tokenizer.
dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

# ------------------------------------------------------------------------------
# Allowed choices for widths and fractional widths.
width_choices = [8, 16, 32]
frac_width_choices = [2, 4, 8]

# ------------------------------------------------------------------------------
# Update the construct_model function to accept a custom precision_choices list.
def construct_model(trial, precision_choices):
    trial_model = deepcopy(base_model)
    # Dictionary to record the decision made per layer.
    precision_decisions = {}

    # For each module (using the full dotted name) in the model…
    for name, layer in trial_model.named_modules():
        if isinstance(layer, nn.Linear):
            # For this run the available types are the ones passed in the list.
            available_types = [cls.__name__ for cls in precision_choices]
            chosen_type_name = trial.suggest_categorical(f"{name}_type", available_types)
            precision_decisions[name] = chosen_type_name

            print(f"Layer {name} set to type: {chosen_type_name}")

            # If full precision is chosen, leave the layer unchanged.
            if chosen_type_name == "Linear":
                continue

            # Otherwise, pick the corresponding quantized class.
            chosen_cls = next(cls for cls in precision_choices if cls.__name__ == chosen_type_name)

            # Prepare common kwargs.
            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }

            # Sample hyperparameters based on the chosen quantized type.
            if chosen_type_name == "LinearInteger":
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_frac_width": trial.suggest_categorical(f"{name}_weight_frac_width", frac_width_choices),
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_frac_width": trial.suggest_categorical(f"{name}_data_in_frac_width", frac_width_choices),
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_frac_width": trial.suggest_categorical(f"{name}_bias_frac_width", frac_width_choices),
                    "floor": False,
                }
            elif chosen_type_name in ["LinearMinifloatDenorm", "LinearMinifloatIEEE"]:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_width": 5,
                    "weight_exponent_bias": 15,
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_width": 5,
                    "data_in_exponent_bias": 15,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_width": 5,
                    "bias_exponent_bias": 15,
                }
            elif chosen_type_name == "LinearLog":
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_bias": 0,
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_bias": 0,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_bias": 0,
                }
            elif chosen_type_name == "LinearBlockFP":
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_width": 5,
                    "weight_exponent_bias": 15,
                    "weight_block_size": [16],  # fixed for now.
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_width": 5,
                    "data_in_exponent_bias": 15,
                    "data_in_block_size": [16],
                    "data_in_skip_first_dim": True,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_width": 5,
                    "bias_exponent_bias": 15,
                    "bias_block_size": [16],
                }
            elif chosen_type_name == "LinearBlockLog":
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_bias_width": 0,
                    "weight_block_size": [16],
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_bias_width": 0,
                    "data_in_block_size": [16],
                    "data_in_skip_first_dim": True,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_bias_width": 0,
                    "bias_block_size": [16],
                }
            elif chosen_type_name == "LinearBinary":
                config = {
                    "weight_stochastic": False,
                    "weight_bipolar": True,
                }
            elif chosen_type_name == "LinearBinaryScaling":
                config = {
                    "data_in_stochastic": False,
                    "bias_stochastic": False,
                    "weight_stochastic": False,
                    "data_in_bipolar": True,
                    "bias_bipolar": True,
                    "weight_bipolar": True,
                    "binary_training": True,
                }
            else:
                config = {}

            # Create the new (quantized) layer and copy the parameters.
            new_layer = chosen_cls(**kwargs, config=config)
            new_layer.weight.data = layer.weight.data.clone()
            if layer.bias is not None:
                new_layer.bias.data = layer.bias.data.clone()
              
            # Propagate metadata if available.
            if hasattr(layer, "meta"):
                new_layer.meta = layer.meta.copy()
            else:
                new_layer.meta = {}

            # Replace the layer in the model.
            deepsetattr(trial_model, name, new_layer)

    # Record the per-layer decisions.
    trial.set_user_attr("precision_decisions", precision_decisions)
    return trial_model

# ------------------------------------------------------------------------------
# (Optional) Graph-building function remains unchanged.
def build_graph_from_model(model, cf_args=None):
    from chop.ir.graph import MaseGraph
    mg = MaseGraph(model, cf_args=cf_args)
    
    print("[INFO] Running init_metadata_analysis_pass()...")
    mg, _ = passes.init_metadata_analysis_pass(mg)
    
    print("[INFO] Running add_common_metadata_analysis_pass()...")
    mg, _ = passes.add_common_metadata_analysis_pass(mg)
    
    return mg

# ------------------------------------------------------------------------------
# Define an objective function that uses a given precision_choices list.
def objective(trial, precision_choices):
    # Construct the model with per-layer decisions.
    model_ = construct_model(trial, precision_choices)
    
    # Request a longer training run to visualize the loss curve.
    trainer = get_trainer(
        model=model_,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=10,  # Set higher to let you see the curve.
    )
    
    # Override some Trainer arguments for logging.
    trainer.args.logging_dir = "./logs"      # Directory where logs will be saved.
    trainer.args.logging_steps = 50            # Log every 50 steps.
    trainer.args.evaluation_strategy = "epoch" # Evaluate at the end of each epoch.
    
    # Train the model.
    trainer.train()
    
    # Optionally evaluate the model.
    eval_results = trainer.evaluate()
    trial.set_user_attr("model", model_)
    trial.set_user_attr("precision_type", "MixedPrecision")
    return eval_results["eval_accuracy"]


    

# ------------------------------------------------------------------------------
# Helper function to run an Optuna study for a given pair (nn.Linear and a candidate).
def run_study_for_pair(precision_choices, n_trials=15, timeout=60*60):
    sampler = TPESampler()
    study = optuna.create_study(direction="maximize", sampler=sampler)
    # Use a lambda to pass our custom precision_choices into the objective.
    study.optimize(lambda trial: objective(trial, precision_choices), n_trials=n_trials, timeout=timeout)

    # Gather and return results.
    results = []
    for t in sorted(study.trials, key=lambda t: t.number):
        results.append({
            "trial_number": t.number,
            "trial_accuracy": t.value,
            "precision_decisions": t.user_attrs.get("precision_decisions", {}),
        })
    df = pd.DataFrame(results)
    return df

# ------------------------------------------------------------------------------
# Main loop: Define the candidate pairs.
# Here we always use nn.Linear as the full precision option and then one quantized type.
# Adjust the candidate list as desired.
candidate_types = [
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    #LinearBlockMinifloat, 
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
]

combined_results = []  # To store results for all pairs.
pair_names = []  # To store names for labeling plots.

# For each candidate, run a study.
for candidate in candidate_types:
    # Create the precision choices list for this run:
    # Full precision is always included.
    pair = [nn.Linear, candidate]
    pair_name = f"Linear+{candidate.__name__}"
    pair_names.append(pair_name)
    print(f"\nRunning study for pair: {pair_name}")
    df_pair = run_study_for_pair(pair, n_trials=2, timeout=60*60)
    df_pair["pair"] = pair_name

    # Save individual results.
    csv_name = f"optuna_results_{pair_name}.csv"
    df_pair.to_csv(csv_name, index=False)
    print(f"Results saved to {csv_name}")

    combined_results.append(df_pair)

# Combine all results into one DataFrame.
df_combined = pd.concat(combined_results, ignore_index=True)
df_combined.to_csv("optuna_combined_results_all_pairs.csv", index=False)
print("Combined results saved to optuna_combined_results_all_pairs.csv")

# ------------------------------------------------------------------------------
# Plotting: Create a plot for each pair showing the cumulative best accuracy over trials.
plt.figure(figsize=(8, 6))
for pair_name in pair_names:
    df_pair = df_combined[df_combined["pair"] == pair_name].sort_values("trial_number")
    trial_nums = df_pair["trial_number"].tolist()
    accuracies = df_pair["trial_accuracy"].tolist()
    cum_best = []
    current_best = -float("inf")
    for acc in accuracies:
        current_best = max(current_best, acc)
        cum_best.append(current_best)
    plt.plot(trial_nums, cum_best, marker="o", label=pair_name)

plt.xlabel("Trial Number")
plt.ylabel("Cumulative Best Accuracy")
plt.title("Optimization Progress by Precision Pair")
plt.legend()
plt.grid(True)
plt.savefig("optuna_progress_all_pairs.png")
plt.show()


INFO     Tokenizing dataset imdb with AutoTokenizer for bert-base-uncased.
[I 2025-02-06 16:45:06,051] A new study created in memory with name: no-name-9e747bf7-226e-41d3-9453-2d9d75c5daef



Running study for pair: Linear+LinearInteger
Layer bert.encoder.layer.0.attention.self.query set to type: LinearInteger
Layer bert.encoder.layer.0.attention.self.key set to type: Linear
Layer bert.encoder.layer.0.attention.self.value set to type: LinearInteger
Layer bert.encoder.layer.0.attention.output.dense set to type: LinearInteger
Layer bert.encoder.layer.0.intermediate.dense set to type: Linear
Layer bert.encoder.layer.0.output.dense set to type: Linear
Layer bert.encoder.layer.1.attention.self.query set to type: Linear
Layer bert.encoder.layer.1.attention.self.key set to type: Linear
Layer bert.encoder.layer.1.attention.self.value set to type: Linear
Layer bert.encoder.layer.1.attention.output.dense set to type: Linear
Layer bert.encoder.layer.1.intermediate.dense set to type: Linear
Layer bert.encoder.layer.1.output.dense set to type: LinearInteger
Layer bert.encoder.layer.2.attention.self.query set to type: LinearInteger
Layer bert.encoder.layer.2.attention.self.key set to ty

/Users/nyalpatel/Library/CloudStorage/OneDrive-Personal/Uni/Imperial MSC/Modules/Advanced Deep Learning/Labs/mase/src/chop/tools/huggingface.py:157: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Step,Training Loss
50,0.439700
100,0.353400
150,0.384600
200,0.335600
250,0.284400
300,0.347800
350,0.349300
400,0.406200
450,0.257900
500,0.363400


[W 2025-02-06 16:46:30,045] Trial 0 failed with parameters: {'bert.encoder.layer.0.attention.self.query_type': 'LinearInteger', 'bert.encoder.layer.0.attention.self.query_weight_width': 16, 'bert.encoder.layer.0.attention.self.query_weight_frac_width': 2, 'bert.encoder.layer.0.attention.self.query_data_in_width': 32, 'bert.encoder.layer.0.attention.self.query_data_in_frac_width': 4, 'bert.encoder.layer.0.attention.self.query_bias_width': 16, 'bert.encoder.layer.0.attention.self.query_bias_frac_width': 4, 'bert.encoder.layer.0.attention.self.key_type': 'Linear', 'bert.encoder.layer.0.attention.self.value_type': 'LinearInteger', 'bert.encoder.layer.0.attention.self.value_weight_width': 8, 'bert.encoder.layer.0.attention.self.value_weight_frac_width': 4, 'bert.encoder.layer.0.attention.self.value_data_in_width': 8, 'bert.encoder.layer.0.attention.self.value_data_in_frac_width': 8, 'bert.encoder.layer.0.attention.self.value_bias_width': 8, 'bert.encoder.layer.0.attention.self.value_bias_fr

KeyboardInterrupt: 

# Task 1 - nn.Linear and Linear Integer

In [ ]:
import random
import dill
from copy import deepcopy
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler

from transformers import AutoModel
from chop.tools import get_tokenized_dataset, get_trainer
from chop.tools.utils import deepsetattr
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockMinifloat,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
    LinearBinaryResidualSign,
)

# ------------------------------------------------------------------------------
# Checkpoints and dataset configuration
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

# Load the base (floating-point) model
model = AutoModel.from_pretrained(checkpoint)
with open("best_tpe_model.pkl", "rb") as f:
    base_model = dill.load(f)

# Get tokenized dataset and tokenizer
dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

# Define the search space for the linear layer types
# (here we use only torch.nn.Linear and LinearInteger, but you can extend this list)
search_space = {
    "linear_layer_choices": [
        torch.nn.Linear,
        LinearInteger,
        # Add other quantized layers if needed.
    ],
}

# ------------------------------------------------------------------------------
def construct_model(trial):
    """
    Constructs a new model for the given trial by replacing linear layers according
    to the trial's suggestions. For each torch.nn.Linear layer, the trial can choose to keep it
    as is or replace it with a quantized layer. For the LinearInteger layer, additional
    hyperparameters for bit-widths are chosen.
    
    A dictionary 'precision_config' is built to record, for each replaced layer,
    the type of quantization used and its configuration.
    """
    # Copy the base model
    trial_model = deepcopy(base_model)
    
    # Dictionary to store precision configuration for each layer that is replaced.
    precision_config = {}

    # Iterate over all named modules in the model.
    for name, layer in trial_model.named_modules():
        if isinstance(layer, torch.nn.Linear):
            # Choose the type for this layer.
            new_layer_cls = trial.suggest_categorical(
                f"{name}_type",
                search_space["linear_layer_choices"],
            )

            # If no quantization is applied, leave the layer unchanged.
            if new_layer_cls == torch.nn.Linear:
                continue

            # Prepare the keyword arguments common to all linear layers.
            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }

            # For LinearInteger layers, choose the bit widths and fractional widths per layer.
            if new_layer_cls == LinearInteger:
                config = {
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", [8, 16, 32]),
                    "data_in_frac_width": trial.suggest_categorical(f"{name}_data_in_frac_width", [2, 4, 8]),
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", [8, 16, 32]),
                    "weight_frac_width": trial.suggest_categorical(f"{name}_weight_frac_width", [2, 4, 8]),
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", [8, 16, 32]),
                    "bias_frac_width": trial.suggest_categorical(f"{name}_bias_frac_width", [2, 4, 8]),
                }
                kwargs["config"] = config
            # You can add more branches here for other quantizer types.

            # Record the precision configuration for this layer.
            precision_config[name] = {
                "quantizer": new_layer_cls.__name__,
                "config": kwargs.get("config", {})
            }

            # Create the new layer and copy over the weights from the original layer.
            new_layer = new_layer_cls(**kwargs)
            new_layer.weight.data = layer.weight.data.clone()
            if layer.bias is not None:
                new_layer.bias.data = layer.bias.data.clone()

            # Replace the layer in the model using deepsetattr (which handles nested attributes)
            deepsetattr(trial_model, name, new_layer)

    # Store the precision configuration in the trial's user attributes.
    trial.set_user_attr("precision_config", precision_config)
    
    return trial_model

# ------------------------------------------------------------------------------
def objective(trial):
    """
    Optuna objective function.
    Constructs the model with quantization hyperparameters chosen by the trial,
    trains the model for a few epochs, evaluates it on the validation set,
    and returns the evaluation accuracy.
    """
    # Construct the quantized model for this trial.
    model = construct_model(trial)

    # Create the trainer. Adjust parameters (such as num_train_epochs) as needed.
    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )

    # Train the model.
    trainer.train()
    # Evaluate the model.
    eval_results = trainer.evaluate()

    # Optionally, store the model with the trial for further inspection.
    trial.set_user_attr("model", model)

    return eval_results["eval_accuracy"]

# ------------------------------------------------------------------------------
# Create the Optuna study using the TPESampler.
sampler = TPESampler()
study = optuna.create_study(
    direction="maximize",
    study_name="bert-tiny-nas-study",
    sampler=sampler,
)

# Run the optimization.
study.optimize(
    objective,
    n_trials=2,  # Adjust the number of trials as needed.
    timeout=60 * 60 * 24,
)

# ------------------------------------------------------------------------------
# After optimization, extract trial results, including precision configuration.
results = []
cumulative_best = -float("inf")
trial_numbers = []
cumulative_max = []

# Sort trials in the order they were executed (by trial.number)
for trial in sorted(study.trials, key=lambda t: t.number):
    cumulative_best = max(cumulative_best, trial.value)
    trial_numbers.append(trial.number)
    cumulative_max.append(cumulative_best)
    
    # Get the stored precision configuration.
    precision_config = trial.user_attrs.get("precision_config", {})
    
    results.append({
        "trial_number": trial.number,
        "trial_accuracy": trial.value,
        "cumulative_best_accuracy": cumulative_best,
        "precision_config": precision_config,
    })

# Save the results to a CSV file.
df = pd.DataFrame(results)
df.to_csv("optuna_results.csv", index=False)
print("Results saved to optuna_results.csv")

# ------------------------------------------------------------------------------
# Plot the number of trials vs. the maximum achieved accuracy up to that point.
plt.figure(figsize=(8, 6))
plt.plot(trial_numbers, cumulative_max, marker="o", linestyle="-")
plt.xlabel("Trial Number")
plt.ylabel("Maximum Achieved Accuracy")
plt.title("Optuna Optimization Progress")
plt.grid(True)
plt.savefig("optuna_progress.png")
plt.show()


# Task 2 - nn.Linear and all other precision types

In [ ]:
import random
import dill
from copy import deepcopy
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler

from transformers import AutoModel
from chop.tools import get_tokenized_dataset, get_trainer
from chop.tools.utils import deepsetattr

# Import supported linear quantizers.
from torch import nn
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockMinifloat,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
    # Exclude LinearBinaryResidualSign.
)
from chop.passes.graph.analysis.quantization.calculate_avg_bits import calculate_avg_bits_mg_analysis_pass

from chop.ir.graph import MaseGraph
import chop.passes as passes

# ------------------------------------------------------------------------------
# Configuration: Checkpoints, dataset, etc.
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

# Load the floating-point base model.
model = AutoModel.from_pretrained(checkpoint)
with open("best_tpe_model.pkl", "rb") as f:
    base_model = dill.load(f)

# Get tokenized dataset and tokenizer.
dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)


# ------------------------------------------------------------------------------
# Allowed choices for widths and fractional widths.
width_choices = [8, 16, 32]
frac_width_choices = [2, 4, 8]

# ------------------------------------------------------------------------------
# Update the construct_model function to accept a custom precision_choices list.
def construct_model(trial, precision_choices):
    trial_model = deepcopy(base_model)
    # Dictionary to record the decision made per layer.
    precision_decisions = {}

    # For each module (using the full dotted name) in the model…
    for name, layer in trial_model.named_modules():
        if isinstance(layer, nn.Linear):
            # For this run the available types are the ones passed in the list.
            available_types = [cls.__name__ for cls in precision_choices]
            chosen_type_name = trial.suggest_categorical(f"{name}_type", available_types)
            precision_decisions[name] = chosen_type_name

            print(f"Layer {name} set to type: {chosen_type_name}")

            # If full precision is chosen, leave the layer unchanged.
            if chosen_type_name == "Linear":
                continue

            # Otherwise, pick the corresponding quantized class.
            chosen_cls = next(cls for cls in precision_choices if cls.__name__ == chosen_type_name)

            # Prepare common kwargs.
            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }

            # Sample hyperparameters based on the chosen quantized type.
            if chosen_type_name == "LinearInteger":
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_frac_width": trial.suggest_categorical(f"{name}_weight_frac_width", frac_width_choices),
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_frac_width": trial.suggest_categorical(f"{name}_data_in_frac_width", frac_width_choices),
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_frac_width": trial.suggest_categorical(f"{name}_bias_frac_width", frac_width_choices),
                    "floor": False,
                }
            elif chosen_type_name in ["LinearMinifloatDenorm", "LinearMinifloatIEEE"]:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_width": 5,
                    "weight_exponent_bias": 15,
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_width": 5,
                    "data_in_exponent_bias": 15,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_width": 5,
                    "bias_exponent_bias": 15,
                }
            elif chosen_type_name == "LinearLog":
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_bias": 0,
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_bias": 0,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_bias": 0,
                }
            elif chosen_type_name == "LinearBlockFP":
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_width": 5,
                    "weight_exponent_bias": 15,
                    "weight_block_size": [16],  # fixed for now.
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_width": 5,
                    "data_in_exponent_bias": 15,
                    "data_in_block_size": [16],
                    "data_in_skip_first_dim": True,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_width": 5,
                    "bias_exponent_bias": 15,
                    "bias_block_size": [16],
                }
            elif chosen_type_name == "LinearBlockMiniFloat":
                config = {
                    # Weight quantization parameters
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_width": 5,
                    "weight_exponent_bias_width": 15,
                    "weight_block_size": [16],  # fixed block size; adjust if needed

                    # Data input quantization parameters
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_width": 5,
                    "data_in_exponent_bias_width": 15,
                    "data_in_block_size": [16],
                    "data_in_skip_first_dim": True,  # or False if you need to quantize the first dimension

                    # Bias quantization parameters
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_width": 5,
                    "bias_exponent_bias_width": 15,
                    "bias_block_size": [16],
                }
            elif chosen_type_name == "LinearBlockLog":
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_bias_width": 0,
                    "weight_block_size": [16],
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_bias_width": 0,
                    "data_in_block_size": [16],
                    "data_in_skip_first_dim": True,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_bias_width": 0,
                    "bias_block_size": [16],
                }
            elif chosen_type_name == "LinearBinary":
                config = {
                    "weight_stochastic": False,
                    "weight_bipolar": True,
                }
            elif chosen_type_name == "LinearBinaryScaling":
                config = {
                    "data_in_stochastic": False,
                    "bias_stochastic": False,
                    "weight_stochastic": False,
                    "data_in_bipolar": True,
                    "bias_bipolar": True,
                    "weight_bipolar": True,
                    "binary_training": True,
                }
            else:
                config = {}

            # Create the new (quantized) layer and copy the parameters.
            new_layer = chosen_cls(**kwargs, config=config)
            new_layer.weight.data = layer.weight.data.clone()
            if layer.bias is not None:
                new_layer.bias.data = layer.bias.data.clone()

            # Propagate metadata if available.
            if hasattr(layer, "meta"):
                new_layer.meta = layer.meta.copy()
            else:
                new_layer.meta = {}

            # Replace the layer in the model.
            deepsetattr(trial_model, name, new_layer)

    # Record the per-layer decisions.
    trial.set_user_attr("precision_decisions", precision_decisions)
    return trial_model

# ------------------------------------------------------------------------------
# (Optional) Graph-building function remains unchanged.
def build_graph_from_model(model, cf_args=None):
    from chop.ir.graph import MaseGraph
    mg = MaseGraph(model, cf_args=cf_args)

    print("[INFO] Running init_metadata_analysis_pass()...")
    mg, _ = passes.init_metadata_analysis_pass(mg)

    print("[INFO] Running add_common_metadata_analysis_pass()...")
    mg, _ = passes.add_common_metadata_analysis_pass(mg)

    return mg

# ------------------------------------------------------------------------------
# Define an objective function that uses a given precision_choices list.
def objective(trial, precision_choices):
    # Construct the model with per-layer decisions.
    model_ = construct_model(trial, precision_choices)
    trainer = get_trainer(
        model=model_,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )
    trainer.train()
    eval_results = trainer.evaluate()
    trial.set_user_attr("model", model_)
    trial.set_user_attr("precision_type", "MixedPrecision")
    return eval_results["eval_accuracy"]



# ------------------------------------------------------------------------------
# Helper function to run an Optuna study for a given pair (nn.Linear and a candidate).
def run_study_for_pair(precision_choices, n_trials=15, timeout=60*60*24):
    sampler = TPESampler()
    study = optuna.create_study(direction="maximize", sampler=sampler)
    # Use a lambda to pass our custom precision_choices into the objective.
    study.optimize(lambda trial: objective(trial, precision_choices), n_trials=n_trials, timeout=timeout)

    # Gather and return results.
    results = []
    for t in sorted(study.trials, key=lambda t: t.number):
        results.append({
            "trial_number": t.number,
            "trial_accuracy": t.value,
            "precision_decisions": t.user_attrs.get("precision_decisions", {}),
        })
    df = pd.DataFrame(results)
    return df

# ------------------------------------------------------------------------------
# Main loop: Define the candidate pairs.
# Here we always use nn.Linear as the full precision option and then one quantized type.
# Adjust the candidate list as desired.
candidate_types = [
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
]

combined_results = []  # To store results for all pairs.
pair_names = []  # To store names for labeling plots.

# For each candidate, run a study.
for candidate in candidate_types:
    # Create the precision choices list for this run:
    # Full precision is always included.
    pair = [nn.Linear, candidate]
    pair_name = f"Linear+{candidate.__name__}"
    pair_names.append(pair_name)
    print(f"\nRunning study for pair: {pair_name}")
    df_pair = run_study_for_pair(pair, n_trials=30, timeout=60*60*24)
    df_pair["pair"] = pair_name

    # Save individual results.
    csv_name = f"optuna_results_{pair_name}.csv"
    df_pair.to_csv(csv_name, index=False)
    print(f"Results saved to {csv_name}")

    combined_results.append(df_pair)

# Combine all results into one DataFrame.
df_combined = pd.concat(combined_results, ignore_index=True)
df_combined.to_csv("optuna_combined_results_all_pairs.csv", index=False)
print("Combined results saved to optuna_combined_results_all_pairs.csv")

# ------------------------------------------------------------------------------
# Plotting: Create a plot for each pair showing the cumulative best accuracy over trials.
plt.figure(figsize=(8, 6))
for pair_name in pair_names:
    df_pair = df_combined[df_combined["pair"] == pair_name].sort_values("trial_number")
    trial_nums = df_pair["trial_number"].tolist()
    accuracies = df_pair["trial_accuracy"].tolist()
    cum_best = []
    current_best = -float("inf")
    for acc in accuracies:
        current_best = max(current_best, acc)
        cum_best.append(current_best)
    plt.plot(trial_nums, cum_best, marker="o", label=pair_name)

plt.xlabel("Trial Number")
plt.ylabel("Cumulative Best Accuracy")
plt.title("Optimization Progress by Precision Pair")
plt.legend()
plt.grid(True)
plt.savefig("optuna_progress_all_pairs.png")
plt.show()


# Task 2 Extra - Changing all layers to specific precision type 

In [ ]:
import random
import dill
from copy import deepcopy
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler

from transformers import AutoModel
from chop.tools import get_tokenized_dataset, get_trainer
from chop.tools.utils import deepsetattr

# Import supported linear quantizers.
from torch import nn
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockMinifloat,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
    # LinearBinaryResidualSign,  # Ignored as per professor's suggestion.
)

# ------------------------------------------------------------------------------
# Configuration: Checkpoints, dataset, etc.
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

# Load the floating-point base model.
model = AutoModel.from_pretrained(checkpoint)
with open("best_tpe_model.pkl", "rb") as f:
    base_model = dill.load(f)

# Get tokenized dataset and tokenizer.
dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

# ------------------------------------------------------------------------------
# Define the list of precision types you want to test.
precision_choices = [
    nn.Linear,  
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
    # Exclude LinearBinaryResidualSign.
]

# Allowed choices for widths and fractional widths.
width_choices = [8, 16, 32]
frac_width_choices = [2, 4, 8]

# ------------------------------------------------------------------------------
def construct_model_fixed(trial, chosen_precision):
    """
    Constructs a new model where the precision type for *all* nn.Linear layers is fixed
    to `chosen_precision` (except for full precision, which is nn.Linear).
    Hyperparameters for widths are sampled for each layer if applicable.
    """
    trial_model = deepcopy(base_model)
    
    for name, layer in trial_model.named_modules():
        if isinstance(layer, nn.Linear):
            # For full precision, do nothing.
            if chosen_precision == nn.Linear:
                continue

            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }
            # Depending on the chosen precision, create a configuration.
            if chosen_precision == LinearInteger:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_frac_width": trial.suggest_categorical(f"{name}_weight_frac_width", frac_width_choices),
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_frac_width": trial.suggest_categorical(f"{name}_data_in_frac_width", frac_width_choices),
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_frac_width": trial.suggest_categorical(f"{name}_bias_frac_width", frac_width_choices),
                    "floor": False,
                }
            elif chosen_precision in [LinearMinifloatDenorm, LinearMinifloatIEEE]:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_width": 5,
                    "weight_exponent_bias": 15,
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_width": 5,
                    "data_in_exponent_bias": 15,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_width": 5,
                    "bias_exponent_bias": 15,
                }
            elif chosen_precision == LinearLog:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_bias": 0,
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_bias": 0,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_bias": 0,
                }
            elif chosen_precision == LinearBlockFP:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_width": 5,
                    "weight_exponent_bias": 15,
                    "weight_block_size": [16],  # kept fixed for now.
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_width": 5,
                    "data_in_exponent_bias": 15,
                    "data_in_block_size": [16],
                    "data_in_skip_first_dim": True,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_width": 5,
                    "bias_exponent_bias": 15,
                    "bias_block_size": [16],
                }
            elif chosen_precision == LinearBlockLog:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_bias_width": 0,
                    "weight_block_size": [16],
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_bias_width": 0,
                    "data_in_block_size": [16],
                    "data_in_skip_first_dim": True,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_bias_width": 0,
                    "bias_block_size": [16],
                }
            elif chosen_precision == LinearBinary:
                config = {
                    "weight_stochastic": False,
                    "weight_bipolar": True,
                }
            elif chosen_precision == LinearBinaryScaling:
                config = {
                    "data_in_stochastic": False,
                    "bias_stochastic": False,
                    "weight_stochastic": False,
                    "data_in_bipolar": True,
                    "bias_bipolar": True,
                    "weight_bipolar": True,
                    "binary_training": True,
                }
            else:
                config = {}

            # Create the new layer.
            new_layer = chosen_precision(**kwargs, config=config)
            new_layer.weight.data = layer.weight.data.clone()
            if layer.bias is not None:
                new_layer.bias.data = layer.bias.data.clone()
            deepsetattr(trial_model, name, new_layer)

    return trial_model

# ------------------------------------------------------------------------------
def objective(trial, chosen_precision):
    # Construct the model with the fixed precision type.
    model = construct_model_fixed(trial, chosen_precision)
    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=1,
    )
    trainer.train()
    eval_results = trainer.evaluate()
    trial.set_user_attr("model", model)
    # Record the fixed precision name as a user attribute.
    trial.set_user_attr("precision_type", chosen_precision.__name__ if chosen_precision != nn.Linear else "FullPrecision")
    return eval_results["eval_accuracy"]

# ------------------------------------------------------------------------------
def run_study_for_precision(chosen_precision, n_trials=5):
    """Runs an Optuna study for a fixed precision type."""
    sampler = TPESampler()
    study_name = f"study_{chosen_precision.__name__}" if chosen_precision != nn.Linear else "study_FullPrecision"
    study = optuna.create_study(direction="maximize", study_name=study_name, sampler=sampler)
    
    # Optimize the study with a lambda that fixes the chosen precision.
    study.optimize(lambda trial: objective(trial, chosen_precision),
                   n_trials=n_trials,
                   timeout=60 * 60 * 24)  # adjust timeout as needed

    # Gather results into a DataFrame.
    results = []
    for t in sorted(study.trials, key=lambda t: t.number):
        results.append({
            "trial_number": t.number,
            "trial_accuracy": t.value,
            "precision_type": t.user_attrs.get("precision_type", "Unknown")
        })
    df = pd.DataFrame(results)
    csv_name = f"optuna_results_{chosen_precision.__name__ if chosen_precision != nn.Linear else 'FullPrecision'}.csv"
    df.to_csv(csv_name, index=False)
    print(f"Results for precision {chosen_precision.__name__ if chosen_precision != nn.Linear else 'FullPrecision'} saved to {csv_name}")
    return df

# ------------------------------------------------------------------------------
# Run studies for each precision type sequentially.
all_results = []  # to store all results for final plotting

for prec in precision_choices:
    print(f"\nRunning study for precision type: {prec.__name__ if prec != nn.Linear else 'FullPrecision'}")
    df_prec = run_study_for_precision(prec, n_trials=5)
    # Append a column to mark the precision type.
    df_prec["precision_type"] = prec.__name__ if prec != nn.Linear else "FullPrecision"
    all_results.append(df_prec)
    
    # (Optional) Plot the cumulative best accuracy for this precision type.
    df_prec = df_prec.sort_values("trial_number")
    trial_nums = df_prec["trial_number"].tolist()
    accuracies = df_prec["trial_accuracy"].tolist()
    cum_best = []
    current_best = -float("inf")
    for acc in accuracies:
        current_best = max(current_best, acc)
        cum_best.append(current_best)
    plt.figure(figsize=(6,4))
    plt.plot(trial_nums, cum_best, marker="o", label=prec.__name__ if prec != nn.Linear else "FullPrecision")
    plt.xlabel("Trial Number")
    plt.ylabel("Cumulative Best Accuracy")
    plt.title(f"Optimization Progress for {prec.__name__ if prec != nn.Linear else 'FullPrecision'}")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"optuna_progress_{prec.__name__ if prec != nn.Linear else 'FullPrecision'}.png")
    plt.show()

# ------------------------------------------------------------------------------
# Combine all results and plot one figure with one curve per precision type.
combined_df = pd.concat(all_results, ignore_index=True)

plt.figure(figsize=(10, 6))
# Group results by precision type.
for precision, group in combined_df.groupby("precision_type"):
    group = group.sort_values("trial_number")
    trial_nums = group["trial_number"].tolist()
    accuracies = group["trial_accuracy"].tolist()

    cum_best = []
    current_best = -float("inf")
    for acc in accuracies:
        current_best = max(current_best, acc)
        cum_best.append(current_best)
    plt.plot(trial_nums, cum_best, marker="o", label=precision)

plt.xlabel("Trial Number")
plt.ylabel("Cumulative Best Accuracy")
plt.title("Optimization Progress by Precision Type (5 Trials Each)")
plt.legend(title="Precision Type")
plt.grid(True)
plt.savefig("optuna_combined_precision_progress.png")
plt.show()

# Save the combined results to CSV.
combined_df.to_csv("optuna_combined_results.csv", index=False)
print("Combined results saved to optuna_combined_results.csv")
 

# Task 2 Extra - Addition of bit cost to precision type search

In [ ]:
import random
import dill
from copy import deepcopy
from pathlib import Path

import torch
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler

from transformers import AutoModel
from chop.tools import get_tokenized_dataset, get_trainer
from chop.tools.utils import deepsetattr

from chop.passes.graph.analysis.quantization.calculate_avg_bits import calculate_avg_bits_mg_analysis_pass

from chop.ir.graph import MaseGraph
import chop.passes as passes

# Import supported linear quantizers.
from torch import nn
from chop.nn.quantized.modules.linear import (
    LinearInteger,
    LinearMinifloatDenorm,
    LinearMinifloatIEEE,
    LinearLog,
    LinearBlockFP,
    LinearBlockMinifloat,
    LinearBlockLog,
    LinearBinary,
    LinearBinaryScaling,
    # LinearBinaryResidualSign,  # Ignored as per professor's suggestion.
)

# ------------------------------------------------------------------------------
# Configuration: Checkpoints, dataset, etc.
checkpoint = "prajjwal1/bert-tiny"
tokenizer_checkpoint = "bert-base-uncased"
dataset_name = "imdb"

# Load the floating-point base model.
model = AutoModel.from_pretrained(checkpoint)
with open("best_tpe_model.pkl", "rb") as f:
    base_model = dill.load(f)

# Get tokenized dataset and tokenizer.
dataset, tokenizer = get_tokenized_dataset(
    dataset=dataset_name,
    checkpoint=tokenizer_checkpoint,
    return_tokenizer=True,
)

# For a Hugging Face DatasetDict where the evaluation split is named "validation"
if "validation" in dataset:
    # Choose the number of evaluation examples you want to use
    eval_subset_size = 10000  # Adjust this number as needed
    # Shuffle (optional) and select the subset
    dataset["validation"] = dataset["validation"].shuffle(seed=42).select(range(eval_subset_size))

# Alternatively, if the split is named "test":
if "test" in dataset:
    eval_subset_size = 10000  # Adjust this number as needed
    dataset["test"] = dataset["test"].shuffle(seed=42).select(range(eval_subset_size))

# ------------------------------------------------------------------------------
# Define the list of precision types you want to test.
precision_choices = [
    # nn.Linear,  
    LinearInteger,
    # LinearMinifloatDenorm,
    # LinearMinifloatIEEE,
    # LinearLog,
    # LinearBlockFP,
    # LinearBlockLog,
    # LinearBinary,
    # LinearBinaryScaling,
    # Exclude LinearBinaryResidualSign.
]

# Allowed choices for widths and fractional widths.
width_choices = [8, 16, 32]
frac_width_choices = [2, 4, 8]

# ------------------------------------------------------------------------------
def construct_model_fixed(trial, chosen_precision):
    """
    Constructs a new model where the precision type for *all* nn.Linear layers is fixed
    to `chosen_precision` (except for full precision, which is nn.Linear).
    Hyperparameters for widths are sampled for each layer if applicable.
    """
    trial_model = deepcopy(base_model)
    
    for name, layer in trial_model.named_modules():
        if isinstance(layer, nn.Linear):
            # For full precision, do nothing.
            if chosen_precision == nn.Linear:
                continue

            kwargs = {
                "in_features": layer.in_features,
                "out_features": layer.out_features,
            }
            # Depending on the chosen precision, create a configuration.
            if chosen_precision == LinearInteger:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_frac_width": trial.suggest_categorical(f"{name}_weight_frac_width", frac_width_choices),
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_frac_width": trial.suggest_categorical(f"{name}_data_in_frac_width", frac_width_choices),
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_frac_width": trial.suggest_categorical(f"{name}_bias_frac_width", frac_width_choices),
                    "floor": False,
                }
            elif chosen_precision in [LinearMinifloatDenorm, LinearMinifloatIEEE]:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_width": 5,
                    "weight_exponent_bias": 15,
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_width": 5,
                    "data_in_exponent_bias": 15,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_width": 5,
                    "bias_exponent_bias": 15,
                }
            elif chosen_precision == LinearLog:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_bias": 0,
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_bias": 0,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_bias": 0,
                }
            elif chosen_precision == LinearBlockFP:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_width": 5,
                    "weight_exponent_bias": 15,
                    "weight_block_size": [16],  # kept fixed for now.
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_width": 5,
                    "data_in_exponent_bias": 15,
                    "data_in_block_size": [16],
                    "data_in_skip_first_dim": True,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_width": 5,
                    "bias_exponent_bias": 15,
                    "bias_block_size": [16],
                }
            elif chosen_precision == LinearBlockLog:
                config = {
                    "weight_width": trial.suggest_categorical(f"{name}_weight_width", width_choices),
                    "weight_exponent_bias_width": 0,
                    "weight_block_size": [16],
                    "data_in_width": trial.suggest_categorical(f"{name}_data_in_width", width_choices),
                    "data_in_exponent_bias_width": 0,
                    "data_in_block_size": [16],
                    "data_in_skip_first_dim": True,
                    "bias_width": trial.suggest_categorical(f"{name}_bias_width", width_choices),
                    "bias_exponent_bias_width": 0,
                    "bias_block_size": [16],
                }
            elif chosen_precision == LinearBinary:
                config = {
                    "weight_stochastic": False,
                    "weight_bipolar": True,
                }
            elif chosen_precision == LinearBinaryScaling:
                config = {
                    "data_in_stochastic": False,
                    "bias_stochastic": False,
                    "weight_stochastic": False,
                    "data_in_bipolar": True,
                    "bias_bipolar": True,
                    "weight_bipolar": True,
                    "binary_training": True,
                }
            else:
                config = {}

            # Create the new layer.
            new_layer = chosen_precision(**kwargs, config=config)
            new_layer.weight.data = layer.weight.data.clone()
            if layer.bias is not None:
                new_layer.bias.data = layer.bias.data.clone()
            deepsetattr(trial_model, name, new_layer)

    return trial_model

# ------------------------------------------------------------------------------
def build_graph_from_model(model, cf_args=None):
    from chop.ir.graph import MaseGraph
    # Create a MaseGraph instance from the given model.
    # cf_args: Optional concrete forward arguments for tracing.
    mg = MaseGraph(model, cf_args=cf_args)

    # Step 3: Run the metadata passes.
    print("[INFO] Running init_metadata_analysis_pass()...")
    mg, _ = passes.init_metadata_analysis_pass(mg)

    print("[INFO] Running add_common_metadata_analysis_pass()...")
    mg, _ = passes.add_common_metadata_analysis_pass(mg)

    return mg

# ------------------------------------------------------------------------------
# Updated objective now accepts chosen_precision and uses construct_model_fixed.
def objective(trial, chosen_precision):
    # Construct the model with the chosen precision.
    model = construct_model_fixed(trial, chosen_precision)

    trainer = get_trainer(
        model=model,
        tokenized_dataset=dataset,
        tokenizer=tokenizer,
        evaluate_metric="accuracy",
        num_train_epochs=0.1,
    )

    trainer.train()
    eval_results = trainer.evaluate()
    
    # Store the raw evaluation accuracy as a user attribute.
    trial.set_user_attr("eval_accuracy", eval_results["eval_accuracy"])

    model = model.cpu()
    # --- Average Bits Analysis ---
    graph = build_graph_from_model(model)
    graph, avg_bit_dict = calculate_avg_bits_mg_analysis_pass(graph, pass_args={})
    model = model.cuda()

    # Store the average bits results as a user attribute.
    trial.set_user_attr("avg_bit_dict", avg_bit_dict)
    # Also store the precision type for later reference.
    trial.set_user_attr("precision_type", chosen_precision.__name__ if chosen_precision != nn.Linear else "FullPrecision")
    print("Average bits analysis:", avg_bit_dict)

    # --- Composite Metric ---
    alpha = 0.0083  # Adjust the scaling factor as needed.
    composite_metric = eval_results["eval_accuracy"] - alpha * (avg_bit_dict['w_avg_bit'] + avg_bit_dict['data_avg_bit'])
    return composite_metric

# ------------------------------------------------------------------------------
def run_study_for_precision(chosen_precision, n_trials=5):
    """Runs an Optuna study for a fixed precision type."""
    sampler = TPESampler()
    study_name = f"study_{chosen_precision.__name__}" if chosen_precision != nn.Linear else "study_FullPrecision"
    study = optuna.create_study(direction="maximize", study_name=study_name, sampler=sampler)
    
    # Optimize the study with a lambda that fixes the chosen precision.
    study.optimize(lambda trial: objective(trial, chosen_precision),
                   n_trials=n_trials,
                   timeout=60 * 60 * 24)  # adjust timeout as needed

    # Print best trial's average bits.
    best_trial = study.best_trial
    print(f"\nBest trial for {chosen_precision.__name__ if chosen_precision != nn.Linear else 'FullPrecision'}:")
    print("Trial Number:", best_trial.number)
    print("Best Composite Metric (Accuracy - alpha*bits):", best_trial.value)
    print("Best trial's avg_bit_dict:", best_trial.user_attrs.get("avg_bit_dict", {}))

    # Gather results into a DataFrame.
    results = []
    # We track both the best composite metric and the best raw evaluation accuracy so far.
    current_best_composite = -float("inf")
    current_best_eval = -float("inf")
    current_best_avg_bits = {"w_avg_bit": None, "data_avg_bit": None}
    current_best_eval_avg_bits = {"w_avg_bit": None, "data_avg_bit": None}
    for t in sorted(study.trials, key=lambda t: t.number):
        composite = t.value
        eval_acc = t.user_attrs.get("eval_accuracy", None)
        if composite is not None and composite > current_best_composite:
            current_best_composite = composite
            current_best_avg_bits = t.user_attrs.get("avg_bit_dict", {"w_avg_bit": None, "data_avg_bit": None})
        if eval_acc is not None and eval_acc > current_best_eval:
            current_best_eval = eval_acc
            current_best_eval_avg_bits = t.user_attrs.get("avg_bit_dict", {"w_avg_bit": None, "data_avg_bit": None})
        results.append({
            "trial_number": t.number,
            "trial_composite_metric": composite,
            "trial_eval_accuracy": eval_acc,
            "precision_type": t.user_attrs.get("precision_type", "Unknown"),
            "current_best_w_avg_bit": current_best_avg_bits.get("w_avg_bit"),
            "current_best_data_avg_bit": current_best_avg_bits.get("data_avg_bit"),
            "current_best_eval_accuracy": current_best_eval,
            "current_best_eval_w_avg_bit": current_best_eval_avg_bits.get("w_avg_bit"),
            "current_best_eval_data_avg_bit": current_best_eval_avg_bits.get("data_avg_bit")
        })
    df = pd.DataFrame(results)
    csv_name = f"optuna_results_{chosen_precision.__name__ if chosen_precision != nn.Linear else 'FullPrecision'}.csv"
    df.to_csv(csv_name, index=False)
    print(f"Results for precision {chosen_precision.__name__ if chosen_precision != nn.Linear else 'FullPrecision'} saved to {csv_name}")
    return df

# ------------------------------------------------------------------------------
# Run studies for each precision type sequentially.
all_results = []  # to store all results for final plotting

for prec in precision_choices:
    print(f"\nRunning study for precision type: {prec.__name__ if prec != nn.Linear else 'FullPrecision'}")
    df_prec = run_study_for_precision(prec, n_trials=10)
    # Append a column to mark the precision type.
    df_prec["precision_type"] = prec.__name__ if prec != nn.Linear else "FullPrecision"
    all_results.append(df_prec)
    
    # (Optional) Plot the cumulative best composite metric (as before).
    df_temp = df_prec.sort_values("trial_number")
    trial_nums = df_temp["trial_number"].tolist()
    comp_metrics = df_temp["trial_composite_metric"].tolist()
    cum_best_comp = []
    current_best_comp = -float("inf")
    for val in comp_metrics:
        current_best_comp = max(current_best_comp, val)
        cum_best_comp.append(current_best_comp)
    plt.figure(figsize=(6,4))
    plt.plot(trial_nums, cum_best_comp, marker="o", label=prec.__name__ if prec != nn.Linear else "FullPrecision")
    plt.xlabel("Trial Number")
    plt.ylabel("Cumulative Best Composite Metric")
    plt.title(f"Optimization Progress (Composite Metric) for {prec.__name__ if prec != nn.Linear else 'FullPrecision'}")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"optuna_progress_{prec.__name__ if prec != nn.Linear else 'FullPrecision'}.png")
    plt.show()
    
    # --- New Plot: Cumulative Best Raw Evaluation Accuracy ---
    df_temp = df_prec.sort_values("trial_number")
    trial_nums = df_temp["trial_number"].tolist()
    eval_accs = df_temp["trial_eval_accuracy"].tolist()
    cum_best_eval = []
    current_best_eval = -float("inf")
    for acc in eval_accs:
        current_best_eval = max(current_best_eval, acc)
        cum_best_eval.append(current_best_eval)
    plt.figure(figsize=(6,4))
    plt.plot(trial_nums, cum_best_eval, marker="o", label=prec.__name__ if prec != nn.Linear else "FullPrecision")
    plt.xlabel("Trial Number")
    plt.ylabel("Cumulative Best Evaluation Accuracy")
    plt.title(f"Best Eval Accuracy Progress for {prec.__name__ if prec != nn.Linear else 'FullPrecision'}")
    plt.legend()
    plt.grid(True)
    plt.savefig(f"optuna_eval_accuracy_progress_{prec.__name__ if prec != nn.Linear else 'FullPrecision'}.png")
    plt.show()

# ------------------------------------------------------------------------------
# Combine all results and plot one figure with one curve per precision type (composite metric).
combined_df = pd.concat(all_results, ignore_index=True)

plt.figure(figsize=(10, 6))
# Group results by precision type.
for precision, group in combined_df.groupby("precision_type"):
    group = group.sort_values("trial_number")
    trial_nums = group["trial_number"].tolist()
    comp_metrics = group["trial_composite_metric"].tolist()

    cum_best_comp = []
    current_best_comp = -float("inf")
    for val in comp_metrics:
        current_best_comp = max(current_best_comp, val)
        cum_best_comp.append(current_best_comp)
    plt.plot(trial_nums, cum_best_comp, marker="o", label=precision)

plt.xlabel("Trial Number")
plt.ylabel("Cumulative Best Composite Metric")
plt.title("Optimization Progress by Precision Type (Composite Metric, 5 Trials Each)")
plt.legend(title="Precision Type")
plt.grid(True)
plt.savefig("optuna_combined_precision_progress.png")
plt.show()

# Combined plot for cumulative best evaluation accuracy for all precision types.
plt.figure(figsize=(10, 6))
for precision, group in combined_df.groupby("precision_type"):
    group = group.sort_values("trial_number")
    trial_nums = group["trial_number"].tolist()
    eval_accs = group["trial_eval_accuracy"].tolist()

    cum_best_eval = []
    current_best_eval = -float("inf")
    for acc in eval_accs:
        current_best_eval = max(current_best_eval, acc)
        cum_best_eval.append(current_best_eval)
    plt.plot(trial_nums, cum_best_eval, marker="o", label=precision)

plt.xlabel("Trial Number")
plt.ylabel("Cumulative Best Evaluation Accuracy")
plt.title("Best Evaluation Accuracy Progress by Precision Type")
plt.legend(title="Precision Type")
plt.grid(True)
plt.savefig("optuna_combined_eval_accuracy_progress.png")
plt.show()

# Save the combined results to CSV.
combined_df.to_csv("optuna_combined_results.csv", index=False)
print("Combined results saved to optuna_combined_results.csv")
